# A critic turns a sheller concept into a more testable design

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmudrc/design-research-workshop/blob/workshop-three-agent-demos/notebooks/peanut_sheller_propose_critic.ipynb)

Take one concept through proposal, critique, and revision. Inspect what the critic flags and check which details the proposer changes.

Run each numbered section in order. A CPU runtime is enough. The workshop backend uses deterministic rules and runs without an API key.

## 1. Set up
Run the install cell once in Colab. If you installed the workshop package locally, start in the next section.

In [ ]:
# @title Install the workshop package { display-mode: "form" }
%pip -q install "design-research-workshop @ git+https://github.com/cmudrc/design-research-workshop.git@workshop-three-agent-demos"

## 2. Pick the concept and review priority

Upload `peanut_concept.json` from the brainstorm to this notebook’s **Files** pane. If the file is absent, this exercise loads the same default concept that the brainstorm exports. Choose which concern the critic should report first.

In [1]:
# @title Load a concept { display-mode: "form" }
import json
from pathlib import Path

from design_research_workshop import workshop_client
from design_research_workshop.peanut import (
    FUNCTION_FIRST, ShellerConcept, run_brainstorm, sheller_pattern, sheller_task,
)

concept_path = Path("peanut_concept.json")
priority = "kernel protection"  # @param ["kernel protection", "cleaning access"]
if concept_path.is_file():
    concept = ShellerConcept(**json.loads(concept_path.read_text(encoding="utf-8")))
    source = concept_path.name
else:
    with workshop_client() as client:
        concept = run_brainstorm(client, FUNCTION_FIRST)[1]
    source = "default function-first brainstorm, idea 2"
print(f"Concept from {source}")
print(f"{concept.feed} → {concept.crack} → {concept.separate}")

Concept from peanut_concept.json
Gravity hopper → Hand-cranked rollers → Hand-shaken sieve


## 3. Run the proposal and critique loop

Before running, name one feature that could protect the kernel and one part you would want to remove for cleaning. The Agents pattern passes the critic's feedback to the proposer and stops when the critic approves or the iteration limit is reached.

In [2]:
max_iterations = 2  # Try 1 to see the original proposal and feedback only.
with workshop_client() as client:
    pattern = sheller_pattern(client, max_iterations=max_iterations)
    result = pattern.run(sheller_task(concept, priority), request_id="workshop-sheller-critique")
print(f"Critique rounds: {result.iterations}")
print(f"Approved: {result.approved}")
print(f"Stopping reason: {result.terminated_reason}")

Critique rounds: 2
Approved: True
Stopping reason: approved


## 4. Read the critique and revision

Read the critic's concerns alongside the proposal it reviewed. Track which details changed while the feed, crack, and separation mechanisms stayed selected.

In [3]:
for event in result.critique_iterations:
    draft = ShellerConcept(**json.loads(event["proposal"]))
    print(f"Round {event['iteration']}")
    print(f"  Kernel protection: {draft.kernel_protection}")
    print(f"  Cleaning access: {draft.cleaning_access}")
    print(f"  Critic: {event['feedback'] or 'Approved'}")

Round 1
  Kernel protection: Fixed clearance with no size adjustment during use.
  Cleaning access: Fixed separation assembly with access from the outlet.
  Critic: Kernel protection needs an adjustable limit for different pod sizes. Cleaning access needs a removable separator and a clear debris path.
Round 2
  Kernel protection: Adjustable roller gap and a compliant roller surface.
  Cleaning access: Unclip the sieve and empty the removable collection tray.
  Critic: Approved


## Discuss

Switch the first concern or load a different mechanism combination. How does the feedback change? The critic checks whether the proposal includes provisions for those requirements. What would a prototype test measure to find out whether the provisions work?